# 🦷 YOLO26 Nano — Green Cloth Surgical Instruments (14 classes)

Detection on **green cloth with shadows** (shadows make bounding hard). Dataset is **COCO Segmentation** from Roboflow (`dataset/train/_annotations.coco.json` + `dataset/valid/_annotations.coco.json`). This notebook converts it to **YOLO format** and trains **YOLO26 nano** (fallback to YOLO11 nano if 26 not yet in ultralytics).

Pipeline: `COCO → YOLO txt + yaml → YOLO26n train → validate → predict`  
All 14 classes use `bbox_margin` crop logic from the classification repo — here YOLO learns the box directly, so no manual length feature is needed.

## 0) Install dependencies

In [ ]:
%pip install -q ultralytics pycocotools opencv-python-headless
import ultralytics; print(ultralytics.__version__)

## 1) Prepare data — upload `dataset.zip` (which extracts to `dataset/`) and run


In [ ]:
DATA_DIR = "/content/dataset"
import os, pathlib, zipfile
from pathlib import Path
print(f"DATA_DIR={DATA_DIR} exists={Path(DATA_DIR).exists()}")
for sp in ["train","valid"]:
    p = Path(DATA_DIR)/sp/"_annotations.coco.json"
    print(sp, "exists" if p.exists() else "MISSING", p)
print(list(Path(DATA_DIR).rglob("*.jpg"))[:3])

# If you uploaded dataset.zip via Files > Upload:
# import zipfile
# with zipfile.ZipFile("/content/dataset.zip") as z:
#     z.extractall("/content")  # -> creates /content/dataset/train/...

# If from Drive:
# from google.colab import drive; drive.mount('/content/drive')
# import shutil; shutil.unpack_archive("/content/drive/MyDrive/dataset.zip", "/content")


## 2) Convert COCO → YOLO (detection)

Writes `dataset_yolo/{images,labels}/{train,valid}/` + `dataset_yolo/dataset.yaml`.  
Mapping: `label = index of sorted class names` (same as `dataset.py` in the classification repo).

In [ ]:
import json, shutil
from pathlib import Path
from collections import defaultdict

def find_coco_root(base: Path) -> Path:
    base = Path(base)
    for b in [base, base/"dataset", Path("/content")/"dataset", Path("/content")/"dataset"/"dataset"]:
        if (b/"train"/"_annotations.coco.json").exists():
            return b
    for q in Path("/content").rglob("_annotations.coco.json"):
        return q.parent.parent
    return base

SRC = find_coco_root(Path(DATA_DIR))
print(f"COCO root: {SRC} -> train exists {(SRC/'train'/'_annotations.coco.json').exists() }")
DST = Path("/content/dataset_yolo")
for sp in ["train", "valid", "test"]:
    dst_sp = "val" if sp == "valid" else sp
    (DST/"images"/dst_sp).mkdir(parents=True, exist_ok=True)
    (DST/"labels"/dst_sp).mkdir(parents=True, exist_ok=True)
    for f in (DST/"images"/dst_sp).glob("*"): f.unlink(missing_ok=True)
    for f in (DST/"labels"/dst_sp).glob("*"): f.unlink(missing_ok=True)

# union 14 classes first (handles missing class in a split)
all_cats = set()
for sp in ["train", "valid", "test"]:
    pp = SRC / sp / "_annotations.coco.json"
    if pp.exists():
        all_cats.update([c["name"] for c in json.load(open(pp, encoding="utf-8"))["categories"]])
cats = sorted(all_cats)
name_to_id = {n:i for i,n in enumerate(cats)}
print(f"unified classes ({len(cats)}):", cats)

def coco_to_yolo(coco_path: Path, split: str):
    dst_split = "val" if split == "valid" else split
    data = json.load(open(coco_path, encoding="utf-8"))
    id_to_name = {c["id"]: c["name"] for c in data["categories"]}
    img_info = {im["id"]: im for im in data["images"]}
    g = defaultdict(list)
    for ann in data["annotations"]:
        g[ann["image_id"]].append(ann)
    for img_id, im in img_info.items():
        w, h = im["width"], im["height"]
        fname = Path(im["file_name"]).name
        src_img = SRC / split / fname
        if not src_img.exists():
            cand = list(SRC.rglob(fname))
            if cand:
                src_img = cand[0]
            else:
                cand = list(Path("/content").rglob(fname))
                if cand:
                    src_img = cand[0]
        if src_img.exists():
            shutil.copy2(src_img, DST/"images"/dst_split/fname)
        txt = DST/"labels"/dst_split/(Path(fname).stem + ".txt")
        lines = []
        for ann in g.get(img_id, []):
            cname = id_to_name[ann["category_id"]]
            cid = name_to_id[cname]
            x,y,bw,bh = ann["bbox"]
            xc = (x + bw/2) / w; yc = (y + bh/2) / h
            bw /= w; bh /= h
            xc, yc, bw, bh = [max(0,min(1,v)) for v in (xc,yc,bw,bh)]
            lines.append(f"{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        txt.write_text("\n".join(lines), encoding="utf-8")

for split in ["train", "valid", "test"]:
    pth = SRC / split / "_annotations.coco.json"
    if pth.exists():
        coco_to_yolo(pth, split)

yaml_text = f"path: {DST.as_posix()}\ntrain: images/train\nval: images/val\nnames:\n"
for i,n in enumerate(cats):
    yaml_text += f"  {i}: {n}\n"
(DST/"dataset.yaml").write_text(yaml_text, encoding="utf-8")
print(open(DST/"dataset.yaml").read())
n_tr = len(list((DST/"images/train").glob("*.jpg")))
n_va = len(list((DST/"images/val").glob("*.jpg")))
print(f"images train={n_tr} val={n_va}")
print(f"labels train={len(list((DST/'labels/train').glob('*.txt')))} val={len(list((DST/'labels/val').glob('*.txt')))}")


## 3) Verify a few labels

In [ ]:
import random, cv2
from pathlib import Path
import matplotlib.pyplot as plt
p = Path("/content/dataset_yolo")
imgs = list((p/"images/train").glob("*.jpg"))
random.seed(0); samp = random.sample(imgs, min(3, len(imgs)))
for im_path in samp:
    img = cv2.cvtColor(cv2.imread(str(im_path)), cv2.COLOR_BGR2RGB)
    h,w = img.shape[:2]
    txt = p/"labels/train"/(im_path.stem + ".txt")
    for line in txt.read_text().strip().splitlines():
        if not line: continue
        cid, xc, yc, bw, bh = map(float, line.split())
        x1 = int((xc - bw/2)*w); y1 = int((yc - bh/2)*h)
        x2 = int((xc + bw/2)*w); y2 = int((yc + bh/2)*h)
        cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(img, str(int(cid)), (x1, max(0,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)
    plt.figure(figsize=(6,6)); plt.imshow(img); plt.axis("off"); plt.show()


## 4) Train YOLO26 nano

T4 15 GB → `imgsz=640 batch=32` is safe. `640` is YOLO native.
If `yolo26n.pt` not in your ultralytics version, falls back to `yolo11n.pt`.

In [ ]:
from ultralytics import YOLO
yaml_path = "/content/dataset_yolo/dataset.yaml"
for w in ["yolo26n.pt", "yolo11n.pt", "yolov8n.pt"]:
    try:
        model = YOLO(w)
        print(f"loaded {w}")
        break
    except Exception as e:
        print(f"{w} not found: {e}")
else:
    raise FileNotFoundError("No YOLO nano weights found")
results = model.train(data=yaml_path, epochs=100, imgsz=640, batch=32, device=0, workers=4, project="/content/runs", name="yolo26n_green", exist_ok=True, amp=True, patience=20, optimizer="auto", cos_lr=True, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=5, translate=0.05, scale=0.05, shear=1, fliplr=0.5, mosaic=0.5)
print("best:", results)


## 5) Validate

In [ ]:
from ultralytics import YOLO
model = YOLO("/content/runs/yolo26n_green/weights/best.pt")
metrics = model.val(data="/content/dataset_yolo/dataset.yaml", imgsz=640, batch=32)
print(metrics)


## 6) Predict on a few val images

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import random
model = YOLO("/content/runs/yolo26n_green/weights/best.pt")
val_imgs = list(Path("/content/dataset_yolo/images/val").glob("*.jpg"))
for p in random.sample(val_imgs, min(3, len(val_imgs))):
    r = model.predict(source=str(p), imgsz=640, conf=0.25, save=False, verbose=False)[0]
    print(p.name, [(model.names[int(c)], float(conf)) for c,conf in zip(r.boxes.cls, r.boxes.conf)] if r.boxes is not None else "no det")
    r.save(filename=f"/content/pred_{p.name}")
    from IPython.display import Image, display
    display(Image(filename=f"/content/pred_{p.name}"))


## 7) Export / Download

In [ ]:
from google.colab import files
files.download("/content/runs/yolo26n_green/weights/best.pt")
